# Cohere Command R+ を使用したRAG 日本語ver

## https://lightning.ai/lightning-ai/studios/rag-using-cohere-command-r

<img src="cohereRAG.png">

##### ↑のANNとは、Approximate Nearest Neighbor: 近似最近傍探索のことで、機械学習の教科書に出てくるkNN(k Nearest Neighbor)の一種です。
##### kNNに対して精度を犠牲にして高速化を実現しています。

In [ ]:
!pip install llama_index.core llama_index.llms.cohere llama_index.embeddings.cohere llama_index.postprocessor.cohere_rerank streamlit llama-index-readers-file

In [1]:
%%writefile app.py
#セルの内容をファイルに書き込みます

# Adapted from https://docs.streamlit.io/knowledge-base/tutorials/build-conversational-apps#build-a-simple-chatbot-gui-with-streaming
import os

import base64
import gc
import random
import tempfile
import time
import uuid

from llama_index.core import Settings
from llama_index.core import VectorStoreIndex, ServiceContext, SimpleDirectoryReader
from llama_index.core import PromptTemplate

from llama_index.llms.cohere import Cohere
from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.postprocessor.cohere_rerank import CohereRerank

import streamlit as st


if "id" not in st.session_state:
    st.session_state.id = uuid.uuid4()
    st.session_state.file_cache = {}

session_id = st.session_state.id
client = None

def reset_chat():
    st.session_state.messages = []
    st.session_state.context = None
    gc.collect()


def display_pdf(file):
    # Opening file from file path

    st.markdown("### PDF Preview")
    base64_pdf = base64.b64encode(file.read()).decode("utf-8")

    # Embedding PDF in HTML
    pdf_display = f"""<iframe src="data:application/pdf;base64,{base64_pdf}" width="400" height="100%" type="application/pdf"
                        style="height:100vh; width:100%"
                    >
                    </iframe>"""

    # Displaying File
    st.markdown(pdf_display, unsafe_allow_html=True)


with st.sidebar:
    st.header(f"Cohere API Keyを設定してください")
    st.link_button("Cohereから入手する 🔗", "https://dashboard.cohere.com/api-keys")
    API_KEY = st.text_input("password", type="password", label_visibility="collapsed")

    uploaded_file = st.file_uploader("`.pdf` ファイルを選択してください", type="pdf")

    if uploaded_file and API_KEY:
        try:
            with tempfile.TemporaryDirectory() as temp_dir:
                file_path = os.path.join(temp_dir, uploaded_file.name)
                
                with open(file_path, "wb") as f:
                    f.write(uploaded_file.getvalue())
                
                file_key = f"{session_id}-{uploaded_file.name}"
                st.write("インデックス化しています...")

                if file_key not in st.session_state.get('file_cache', {}):

                    if os.path.exists(temp_dir):
                            loader = SimpleDirectoryReader(
                                input_dir = temp_dir,
                                required_exts=[".pdf"],
                                recursive=True
                            )
                    else:    
                        st.error('アップロードいただいたファイルが見つかりません。確認をお願いします。')
                        st.stop()
                    
                    docs = loader.load_data()

                    # setup llm & embedding model
                    #言語モデル
                    llm = Cohere(api_key=API_KEY, model="command-r-plus")
                    
                    #埋め込みモデル
                    embed_model = CohereEmbedding(
                        cohere_api_key=API_KEY,
                        model_name="embed-multilingual-v3.0",
                        input_type="search_query",
                    )
                    
                    #リランク
                    cohere_rerank = CohereRerank(
                        model='rerank-multilingual-v3.0',
                        api_key=API_KEY,
                    )

                    # インデックス化
                    Settings.embed_model = embed_model
                    index = VectorStoreIndex.from_documents(docs, show_progress=True)

                    # クエリエンジンの組み立て
                    Settings.llm = llm
                    query_engine = index.as_query_engine(streaming=True, node_postprocessors=[cohere_rerank])

                    # ====== プロンプト テンプレート (Prompt Template:上図参照)======
                    qa_prompt_tmpl_str = (
                    "コンテキスト情報は次のとおりです。\n"
                    "---------------------\n"
                    "{context_str}\n"
                    "---------------------\n"
                    "与えられたコンテキスト情報を元にステップバイステップで考えて簡潔に質問に回答してください。もし不明な場合は、回答を作り出そうとせず「わかりません」と答えてください。\n"
                    "質問: {query_str}\n"
                    "答え: "
                    )
                    qa_prompt_tmpl = PromptTemplate(qa_prompt_tmpl_str)

                    query_engine.update_prompts(
                        {"response_synthesizer:text_qa_template": qa_prompt_tmpl}
                    )
                    
                    st.session_state.file_cache[file_key] = query_engine
                else:
                    query_engine = st.session_state.file_cache[file_key]

                # Inform the user that the file is processed and Display the PDF uploaded
                st.success("準備が整いました")
                display_pdf(uploaded_file)
        except Exception as e:
            st.error(f"エラーが発生しました: {e}")
            st.stop()     

col1, col2 = st.columns([6, 1])

with col1:
    st.header(f"⌘ R+ を使った文書の解析")

with col2:
    st.button("Clear ↺", on_click=reset_chat)

# Initialize chat history
if "messages" not in st.session_state:
    reset_chat()


# Display chat messages from history on app rerun
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])


# Accept user input
if prompt := st.chat_input("質問をどうぞ?"):
    # Add user message to chat history
    st.session_state.messages.append({"role": "user", "content": prompt})
    # Display user message in chat message container
    with st.chat_message("user"):
        st.markdown(prompt)

    # Display assistant response in chat message container
    with st.chat_message("assistant"):
        message_placeholder = st.empty()
        full_response = ""
        
        # Simulate stream of response with milliseconds delay
        streaming_response = query_engine.query(prompt)
        
        for chunk in streaming_response.response_gen:
            full_response += chunk
            message_placeholder.markdown(full_response + "▌")

        # full_response = query_engine.query(prompt)

        message_placeholder.markdown(full_response)
        # st.session_state.context = ctx

    # Add assistant response to chat history
    st.session_state.messages.append({"role": "assistant", "content": full_response})

Writing app.py


In [ ]:
!streamlit run ./app.py & sleep 3 && npx -y localtunnel --port 8501